In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [ ]:
from pyspark.sql.functions import *

In [ ]:
from pyspark.sql.window import Window

In [ ]:
df = spark.read.csv("employee.csv", header=True, inferSchema=True)

In [ ]:
df.show()

+---+-----+------+-------------+----------+----------+
| id| name|salary|         city|department|     phone|
+---+-----+------+-------------+----------+----------+
|  0|Alice| 60000|     New York|        HR|1234567890|
|  2|  Bob| 75000|  Los Angeles|   Finance|2345678901|
|  3|Alice| 50000|      Chicago|        IT|3456789012|
|  4|  Bob| 90000|     New York| Marketing|4567890123|
|  5|  Eve| 40000|      Houston|        HR|5678901234|
|  6|Frank| 80000|San Francisco|        IT|6789012345|
|  7|  Bob| 70000|      Chicago|   Finance|7890123456|
|  8| Hank| 45000|     New York|        IT|8901234567|
|  9|  Ivy| 95000|  Los Angeles| Marketing|9012345678|
| 10| Jack| 55000|      Houston|        HR| 123456789|
| 11|Karen| 62000|San Francisco|   Finance|1231231234|
| 12|  Eve| 72000|      Chicago|        IT|2342342345|
| 13|  Mia| 83000|     New York| Marketing|3453453456|
| 14|  Eve| 67000|  Los Angeles|        HR|4564564567|
| 15|Oscar| 47000|      Houston|        IT|5675675678|
| 16| Paul

row number within each dept


In [ ]:
window_spec = Window.partitionBy("department").orderBy("salary")
df.withColumn("row_number",row_number().over(window_spec)).show()

+---+-----+------+-------------+----------+----------+----------+
| id| name|salary|         city|department|     phone|row_number|
+---+-----+------+-------------+----------+----------+----------+
| 17|Quinn| 49000|     New York|   Finance|7897897890|         1|
| 11|Karen| 62000|San Francisco|   Finance|1231231234|         2|
|  7|  Bob| 70000|      Chicago|   Finance|7890123456|         3|
|  2|  Bob| 75000|  Los Angeles|   Finance|2345678901|         4|
| 19|Steve| 85000|      Chicago|   Finance|9019019012|         5|
|  5|  Eve| 40000|      Houston|        HR|5678901234|         1|
| 10| Jack| 55000|      Houston|        HR| 123456789|         2|
|  0|Alice| 60000|     New York|        HR|1234567890|         3|
| 14|  Eve| 67000|  Los Angeles|        HR|4564564567|         4|
| 18|  Eve| 71000|  Los Angeles|        HR|8908908901|         5|
|  8| Hank| 45000|     New York|        IT|8901234567|         1|
| 15|Oscar| 47000|      Houston|        IT|5675675678|         2|
|  3|Alice

2. Rank Employees by Salary

In [ ]:
window_spec = Window.orderBy("Salary")
df.withColumn("rank",dense_rank().over(window_spec)).show()

+---+-----+------+-------------+----------+----------+----+
| id| name|salary|         city|department|     phone|rank|
+---+-----+------+-------------+----------+----------+----+
|  5|  Eve| 40000|      Houston|        HR|5678901234|   1|
|  8| Hank| 45000|     New York|        IT|8901234567|   2|
| 15|Oscar| 47000|      Houston|        IT|5675675678|   3|
| 17|Quinn| 49000|     New York|   Finance|7897897890|   4|
|  3|Alice| 50000|      Chicago|        IT|3456789012|   5|
| 10| Jack| 55000|      Houston|        HR| 123456789|   6|
| 20|Alice| 58000|      Houston|        IT| 120120123|   7|
|  0|Alice| 60000|     New York|        HR|1234567890|   8|
| 11|Karen| 62000|San Francisco|   Finance|1231231234|   9|
| 14|  Eve| 67000|  Los Angeles|        HR|4564564567|  10|
|  7|  Bob| 70000|      Chicago|   Finance|7890123456|  11|
| 18|  Eve| 71000|  Los Angeles|        HR|8908908901|  12|
| 12|  Eve| 72000|      Chicago|        IT|2342342345|  13|
|  2|  Bob| 75000|  Los Angeles|   Finan

Highest Paid Employee in Every Department

In [ ]:
window_spec=Window.partitionBy("department").orderBy("salary")
df.withColumn("rank",dense_rank().over(window_spec)).filter(col("rank")==1).select("name","salary","department").show()

+-----+------+----------+
| name|salary|department|
+-----+------+----------+
|Quinn| 49000|   Finance|
|  Eve| 40000|        HR|
| Hank| 45000|        IT|
|  Mia| 83000| Marketing|
+-----+------+----------+



Running Salary Total Department-wise

In [ ]:
window_spec=Window.partitionBy("department").orderBy(col("salary").desc())\
.rowsBetween(Window.unboundedPreceding
                                                                                              ,Window.currentRow)
df.withColumn("running_salary",
              sum("salary").over(window_spec)).show()

+---+-----+------+-------------+----------+----------+--------------+
| id| name|salary|         city|department|     phone|running_salary|
+---+-----+------+-------------+----------+----------+--------------+
| 19|Steve| 85000|      Chicago|   Finance|9019019012|         85000|
|  2|  Bob| 75000|  Los Angeles|   Finance|2345678901|        160000|
|  7|  Bob| 70000|      Chicago|   Finance|7890123456|        230000|
| 11|Karen| 62000|San Francisco|   Finance|1231231234|        292000|
| 17|Quinn| 49000|     New York|   Finance|7897897890|        341000|
| 18|  Eve| 71000|  Los Angeles|        HR|8908908901|         71000|
| 14|  Eve| 67000|  Los Angeles|        HR|4564564567|        138000|
|  0|Alice| 60000|     New York|        HR|1234567890|        198000|
| 10| Jack| 55000|      Houston|        HR| 123456789|        253000|
|  5|  Eve| 40000|      Houston|        HR|5678901234|        293000|
|  6|Frank| 80000|San Francisco|        IT|6789012345|         80000|
| 12|  Eve| 72000|  

dept avg sal


In [ ]:
window_spec = Window.partitionBy("department")
result = df.withColumn("avg_sal",avg("salary").over(window_spec))
result.show()

+---+-----+------+-------------+----------+----------+------------------+
| id| name|salary|         city|department|     phone|           avg_sal|
+---+-----+------+-------------+----------+----------+------------------+
|  2|  Bob| 75000|  Los Angeles|   Finance|2345678901|           68200.0|
|  7|  Bob| 70000|      Chicago|   Finance|7890123456|           68200.0|
| 11|Karen| 62000|San Francisco|   Finance|1231231234|           68200.0|
| 17|Quinn| 49000|     New York|   Finance|7897897890|           68200.0|
| 19|Steve| 85000|      Chicago|   Finance|9019019012|           68200.0|
|  0|Alice| 60000|     New York|        HR|1234567890|           58600.0|
|  5|  Eve| 40000|      Houston|        HR|5678901234|           58600.0|
| 10| Jack| 55000|      Houston|        HR| 123456789|           58600.0|
| 14|  Eve| 67000|  Los Angeles|        HR|4564564567|           58600.0|
| 18|  Eve| 71000|  Los Angeles|        HR|8908908901|           58600.0|
|  3|Alice| 50000|      Chicago|      

Difference from Department Average

In [ ]:
window_spec = Window.partitionBy("department")
res = df.withColumn("dept_avg",avg("salary").over(window_spec))
result = res.withColumn("difference",abs(res.salary-res.dept_avg))
result.show()

+---+-----+------+-------------+----------+----------+------------------+------------------+
| id| name|salary|         city|department|     phone|          dept_avg|        difference|
+---+-----+------+-------------+----------+----------+------------------+------------------+
|  2|  Bob| 75000|  Los Angeles|   Finance|2345678901|           68200.0|            6800.0|
|  7|  Bob| 70000|      Chicago|   Finance|7890123456|           68200.0|            1800.0|
| 11|Karen| 62000|San Francisco|   Finance|1231231234|           68200.0|            6200.0|
| 17|Quinn| 49000|     New York|   Finance|7897897890|           68200.0|           19200.0|
| 19|Steve| 85000|      Chicago|   Finance|9019019012|           68200.0|           16800.0|
|  0|Alice| 60000|     New York|        HR|1234567890|           58600.0|            1400.0|
|  5|  Eve| 40000|      Houston|        HR|5678901234|           58600.0|           18600.0|
| 10| Jack| 55000|      Houston|        HR| 123456789|           58600

Previous Employee Salary dept wise




In [ ]:
window_spec = Window.partitionBy("department").orderBy("salary")
result=df.withColumn("previous_sal",lag("salary",).over(window_spec))
result.show()

+---+-----+------+-------------+----------+----------+------------+
| id| name|salary|         city|department|     phone|previous_sal|
+---+-----+------+-------------+----------+----------+------------+
| 17|Quinn| 49000|     New York|   Finance|7897897890|        NULL|
| 11|Karen| 62000|San Francisco|   Finance|1231231234|        NULL|
|  7|  Bob| 70000|      Chicago|   Finance|7890123456|       49000|
|  2|  Bob| 75000|  Los Angeles|   Finance|2345678901|       62000|
| 19|Steve| 85000|      Chicago|   Finance|9019019012|       70000|
|  5|  Eve| 40000|      Houston|        HR|5678901234|        NULL|
| 10| Jack| 55000|      Houston|        HR| 123456789|        NULL|
|  0|Alice| 60000|     New York|        HR|1234567890|       40000|
| 14|  Eve| 67000|  Los Angeles|        HR|4564564567|       55000|
| 18|  Eve| 71000|  Los Angeles|        HR|8908908901|       60000|
|  8| Hank| 45000|     New York|        IT|8901234567|        NULL|
| 15|Oscar| 47000|      Houston|        IT|56756

salary diff from Previous Employee Salary dept wise

In [ ]:
windowSpec = Window.partitionBy("department") \
                   .orderBy("salary")

result = df.withColumn(
    "previous_salary",
    lag("salary").over(windowSpec)
).withColumn(
    "salary_difference",
    col("salary") - col("previous_salary")
)

result.show()

+---+-----+------+-------------+----------+----------+---------------+-----------------+
| id| name|salary|         city|department|     phone|previous_salary|salary_difference|
+---+-----+------+-------------+----------+----------+---------------+-----------------+
| 17|Quinn| 49000|     New York|   Finance|7897897890|           NULL|             NULL|
| 11|Karen| 62000|San Francisco|   Finance|1231231234|          49000|            13000|
|  7|  Bob| 70000|      Chicago|   Finance|7890123456|          62000|             8000|
|  2|  Bob| 75000|  Los Angeles|   Finance|2345678901|          70000|             5000|
| 19|Steve| 85000|      Chicago|   Finance|9019019012|          75000|            10000|
|  5|  Eve| 40000|      Houston|        HR|5678901234|           NULL|             NULL|
| 10| Jack| 55000|      Houston|        HR| 123456789|          40000|            15000|
|  0|Alice| 60000|     New York|        HR|1234567890|          55000|             5000|
| 14|  Eve| 67000|  L

top 2 employees by salary in each dept

In [ ]:
window_spec = Window.partitionBy("department").orderBy(col("salary").desc())
df.withColumn("rn",row_number().over(window_spec)).filter(col("rn")<=2).show()

+---+-----+------+-------------+----------+----------+---+
| id| name|salary|         city|department|     phone| rn|
+---+-----+------+-------------+----------+----------+---+
| 19|Steve| 85000|      Chicago|   Finance|9019019012|  1|
|  2|  Bob| 75000|  Los Angeles|   Finance|2345678901|  2|
| 18|  Eve| 71000|  Los Angeles|        HR|8908908901|  1|
| 14|  Eve| 67000|  Los Angeles|        HR|4564564567|  2|
|  6|Frank| 80000|San Francisco|        IT|6789012345|  1|
| 12|  Eve| 72000|      Chicago|        IT|2342342345|  2|
|  9|  Ivy| 95000|  Los Angeles| Marketing|9012345678|  1|
| 16| Paul| 93000|San Francisco| Marketing|6786786789|  2|
+---+-----+------+-------------+----------+----------+---+

